# Sokoban A* Solver

This notebook implements an A* search algorithm to solve Sokoban puzzles. It uses Manhattan distance as a heuristic and includes deadlock detection to optimize the search process.

In [1]:
import sys
import os
from pathlib import Path
import heapq
import time
import pandas as pd

# Ensure Tool_Creation is on sys.path
# Assuming memory structure: project/CSE291A_Project/sokoban_astar.ipynb
ROOT = Path(".").resolve().parent / "CSE291A_Project" / "Tool_Creation"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    from mcts.games.sokoban import Sokoban, SokobanState
    print("Successfully imported Sokoban components.")
except ImportError as e:
    print(f"Import error: {e}")
    print(f"Current sys.path: {sys.path}")

Successfully imported Sokoban components.


## A* Implementation

In [2]:
def manhattan_heuristic(state: SokobanState):
    """
    Heuristic function: sum of Manhattan distances from each box to its nearest target.
    """
    return state.total_box_distance()

def a_star_search(initial_state: SokobanState, time_limit: float = 30.0):
    """
    A* search algorithm for Sokoban.
    Returns: (path, steps, solved, elapsed_time)
    """
    start_time = time.time()
    
    # Priority queue: (f_score, g_score, state_key, current_state, path)
    pq = []
    initial_f = manhattan_heuristic(initial_state)
    heapq.heappush(pq, (initial_f, 0, initial_state.state_key(), initial_state, []))
    
    visited = {initial_state.state_key(): 0}
    
    while pq:
        if time.time() - start_time > time_limit:
            break
            
        f, g, key, state, path = heapq.heappop(pq)
        
        if state._is_solved():
            return path, state.steps, True, time.time() - start_time
            
        if g >= initial_state.max_steps:
            continue
            
        for action in state.legal_actions():
            next_state = state.clone()
            next_state.apply_action(action)
            
            next_key = next_state.state_key()
            next_g = g + 1
            
            if next_key not in visited or next_g < visited[next_key]:
                if next_state._is_deadlocked():
                    continue
                    
                visited[next_key] = next_g
                h = manhattan_heuristic(next_state)
                heapq.heappush(pq, (next_g + h, next_g, next_key, next_state, path + [action]))
                
    return [], initial_state.max_steps, False, time.time() - start_time

## Evaluation and Reporting

In [3]:
def run_evaluation(levels, max_steps=200):
    results = []
    for level_name in levels:
        print(f"Evaluating {level_name}...")
        game = Sokoban(level_name, max_steps=max_steps)
        initial_state = game.new_initial_state()
        
        path, steps, solved, elapsed = a_star_search(initial_state)
        
        results.append({
            "Level": level_name,
            "Solve%": f"{100 if solved else 0}%",
            "AvgRet": f"{1.0 if solved else 0.0:.3f}",
            "Steps": float(steps if solved else max_steps),
            "Time (s)": f"{elapsed:.2f}s"
        })
    return results

## Final Results

Running A* on the specified levels.

In [6]:
levels_to_eval = ["level1", "level2", "level3", "level4", "level5","level6", "level7", "level8", "level9", "level10"]
results = run_evaluation(levels_to_eval)

print(f"\n{'Level':<10} {'A* Solve%':>12} {'A* AvgRet':>12} {'A* Steps':>11}")
print("─" * 50)
for res in results:
    print(f"{res['Level']:<10} {res['Solve%']:>11} {res['AvgRet']:>12} {res['Steps']:>11.1f}")

Evaluating level1...
Evaluating level2...
Evaluating level3...
Evaluating level4...
Evaluating level5...
Evaluating level6...
Evaluating level7...
Evaluating level8...
Evaluating level9...
Evaluating level10...

Level         A* Solve%    A* AvgRet    A* Steps
──────────────────────────────────────────────────
level1            100%        1.000         1.0
level2            100%        1.000         3.0
level3            100%        1.000        14.0
level4            100%        1.000         8.0
level5            100%        1.000        15.0
level6            100%        1.000        41.0
level7            100%        1.000        29.0
level8            100%        1.000        26.0
level9            100%        1.000        20.0
level10           100%        1.000        50.0
